# SQL Log Anomaly Detection

This project inspects the workspace for available datasets, finds that the current workspace contains access logs but no SQL-specific log dataset, and then creates a synthetic SQL-log dataset that mirrors realistic anomalous database access patterns. The notebook builds a reproducible Isolation Forest pipeline, evaluates it on a held-out test split, and exports model artifacts and research plots.

## Objectives
- Inspect available workspace datasets.
- Create or select a SQL log dataset with labeled anomalies when needed.
- Clean and preprocess the log data.
- Engineer activity-aware features such as query frequency, session timing, table sensitivity, and data volume.
- Train and evaluate an `IsolationForest` model.
- Save model artifacts and outputs for auditing and explanation.


In [1]:
import json
import sqlite3
from pathlib import Path

import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.metrics import (accuracy_score, average_precision_score, classification_report,
                             confusion_matrix, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

print('Environment ready.')


Environment ready.


## 3. Dataset acquisition

The workspace is checked for candidate datasets. In this case, a SQLite access-log database is present, but it does not represent SQL queries. We therefore create a labeled synthetic SQL log dataset that captures normal database usage and intentionally suspicious activity patterns such as after-hours access, sensitive-table browsing, large row retrieval, and elevated access frequency.


In [2]:
workspace = Path.cwd()
print(f'Workspace root: {workspace}')

for pattern in ['*.db', '*.csv', '*.parquet']:
    matches = sorted(workspace.glob(pattern))
    print(f'{pattern}: {matches}')

# Inspect SQLite databases in the workspace for available datasets
for db_path in sorted(workspace.glob('*.db')):
    print(f'Inspecting SQLite database: {db_path.name}')
    conn = sqlite3.connect(db_path)
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
    print(tables)
    for table in tables['name'].tolist():
        try:
            schema = pd.read_sql_query(f'PRAGMA table_info({table})', conn)
            print(f' - {table}: {schema.to_dict(orient="records")[:5]}')
        except Exception as exc:
            print(f'Unable to read schema for {table}: {exc}')
    conn.close()

# Synthetic SQL-log dataset generation.
seed = 42
rng = np.random.default_rng(seed)

users = [f'user_{i:02d}' for i in range(1, 31)]
query_types = ['SELECT', 'INSERT', 'UPDATE', 'DELETE', 'MERGE', 'ALTER']
tables = ['employees', 'orders', 'customers', 'payments', 'inventory', 'logs', 'financials', 'hr_records', 'audit_events', 'sessions']

df = pd.DataFrame({
    'event_id': np.arange(1, 7001),
    'timestamp': pd.to_datetime('2024-01-01') + pd.to_timedelta(rng.integers(0, 90 * 24 * 60, size=7000), unit='min'),
    'user_id': rng.choice(users, size=7000),
    'database_name': rng.choice(['sales_db', 'finance_db', 'hr_db', 'analytics_db'], size=7000),
    'query_type': rng.choice(query_types, size=7000),
    'table_name': rng.choice(tables, size=7000),
    'query_length': rng.integers(80, 2200, size=7000),
    'rows_returned': rng.integers(10, 2000, size=7000),
    'duration_ms': rng.integers(50, 8500, size=7000),
    'bytes_transferred': rng.integers(4000, 600000, size=7000),
    'privilege_level': rng.choice(['read', 'write', 'admin'], size=7000, p=[0.55, 0.30, 0.15]),
    'sensitive_table': rng.choice([0, 1], size=7000, p=[0.82, 0.18]),
    'num_join_tables': rng.integers(0, 4, size=7000),
    'error_count': rng.integers(0, 3, size=7000),
    'is_successful': rng.choice([0, 1], size=7000, p=[0.92, 0.08]),
})

# Add user-level query frequency and time-based features.
df['hour_of_day'] = df['timestamp'].dt.hour
df['is_weekend'] = (df['timestamp'].dt.dayofweek >= 5).astype(int)
df['after_hours'] = ((df['hour_of_day'] < 7) | (df['hour_of_day'] >= 22)).astype(int)
user_query_frequency = df.groupby('user_id').size().to_dict()
df['query_frequency'] = df['user_id'].map(user_query_frequency)

df['anomaly_label'] = 0
anomaly_index = rng.choice(df.index, size=int(len(df) * 0.08), replace=False)
df.loc[anomaly_index, 'anomaly_label'] = 1

# Make anomalies structurally different.
df.loc[anomaly_index, 'query_type'] = rng.choice(['SELECT', 'DELETE', 'MERGE', 'ALTER'], size=len(anomaly_index))
df.loc[anomaly_index, 'table_name'] = rng.choice(['financials', 'hr_records', 'audit_events', 'customers'], size=len(anomaly_index))
df.loc[anomaly_index, 'query_length'] = rng.integers(3000, 15000, size=len(anomaly_index))
df.loc[anomaly_index, 'rows_returned'] = rng.integers(250000, 5000000, size=len(anomaly_index))
df.loc[anomaly_index, 'duration_ms'] = rng.integers(5000, 80000, size=len(anomaly_index))
df.loc[anomaly_index, 'bytes_transferred'] = rng.integers(500000, 35000000, size=len(anomaly_index))
df.loc[anomaly_index, 'privilege_level'] = 'admin'
df.loc[anomaly_index, 'sensitive_table'] = 1
df.loc[anomaly_index, 'after_hours'] = 1
df.loc[anomaly_index, 'is_successful'] = 0
df.loc[anomaly_index, 'error_count'] = rng.integers(1, 6, size=len(anomaly_index))
df.loc[anomaly_index, 'query_frequency'] = df.loc[anomaly_index, 'query_frequency'] + rng.integers(30, 200, size=len(anomaly_index))

df = df.sort_values('timestamp').reset_index(drop=True)
output_csv = workspace / 'synthetic_sql_logs.csv'
df.to_csv(output_csv, index=False)
print(f'Synthetic SQL log dataset saved to {output_csv}')
print(df.shape)
display(df.head())
print('Class distribution:')
print(df['anomaly_label'].value_counts(normalize=True).to_string())


Workspace root: C:\Users\POOJASRI M\Downloads\pavithra\pavithra
*.db: [WindowsPath('C:/Users/POOJASRI M/Downloads/pavithra/pavithra/access_logs (1).db'), WindowsPath('C:/Users/POOJASRI M/Downloads/pavithra/pavithra/access_logs.db')]
*.csv: [WindowsPath('C:/Users/POOJASRI M/Downloads/pavithra/pavithra/cert_preprocessed_ml_comparison.csv'), WindowsPath('C:/Users/POOJASRI M/Downloads/pavithra/pavithra/synthetic_sql_logs.csv')]
*.parquet: [WindowsPath('C:/Users/POOJASRI M/Downloads/pavithra/pavithra/feature_matrix_v1.parquet')]
Inspecting SQLite database: access_logs (1).db
                 name
0           employees
1  file_access_events
2        login_events
3      session_labels
4     sqlite_sequence
 - employees: [{'cid': 0, 'name': 'employee_id', 'type': 'INTEGER', 'notnull': 0, 'dflt_value': None, 'pk': 1}, {'cid': 1, 'name': 'name', 'type': 'TEXT', 'notnull': 0, 'dflt_value': None, 'pk': 0}, {'cid': 2, 'name': 'department', 'type': 'TEXT', 'notnull': 0, 'dflt_value': None, 'pk': 0},

,event_id,timestamp,user_id,database_name,query_type,table_name,query_length,rows_returned,duration_ms,bytes_transferred,privilege_level,sensitive_table,num_join_tables,error_count,is_successful,hour_of_day,is_weekend,after_hours,query_frequency,anomaly_label
0,5083,2024-01-01 00:00:00,user_24,analytics_db,DELETE,logs,801,81,5308,495260,read,0,0,0,0,0,0,1,224,0
1,2316,2024-01-01 01:07:00,user_04,sales_db,DELETE,payments,225,479,3661,561319,admin,0,1,0,0,1,0,1,243,0
2,3674,2024-01-01 01:13:00,user_21,hr_db,SELECT,orders,1763,1809,6568,387273,write,1,0,0,0,1,0,1,216,0
3,6302,2024-01-01 01:46:00,user_20,finance_db,MERGE,audit_events,508,690,2898,141563,write,0,2,2,0,1,0,1,257,0
4,5760,2024-01-01 02:08:00,user_14,finance_db,ALTER,financials,1053,1986,6099,123872,read,0,1,1,0,2,0,1,250,0


Class distribution:
anomaly_label
0    0.92
1    0.08


## 4. Exploratory data analysis

The generated synthetic SQL log dataset includes both normal and suspicious activity patterns, and it will be used for reproducible model training and evaluation. We inspect the dataset shape, data types, missingness, and class balance before training.


In [3]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

output_dir = workspace / 'outputs'
charts_dir = output_dir / 'charts'
charts_dir.mkdir(parents=True, exist_ok=True)

sns.histplot(df['query_length'], bins=30, kde=True, ax=axes[0])
axes[0].set_title('Query length distribution')

sns.histplot(df['rows_returned'], bins=30, kde=True, ax=axes[1])
axes[1].set_title('Rows returned distribution')

sns.histplot(df['duration_ms'], bins=30, kde=True, ax=axes[2])
axes[2].set_title('Query duration distribution')

sns.countplot(data=df, x='query_type', ax=axes[3])
axes[3].set_title('Query type counts')
plt.tight_layout()
plt.savefig(charts_dir / 'eda_overview.png', dpi=300, bbox_inches='tight')
plt.close()
print('Saved EDA plot to outputs/charts/eda_overview.png')


Saved EDA plot to outputs/charts/eda_overview.png


## 5. Data preprocessing

We clean the dataset, ensure no target leakage, and prepare a reproducible feature matrix. The target column is kept separate from the predictive features, and preprocessing is restricted to the training data only when fitting the model.


In [4]:
output_dir = workspace / 'outputs'
charts_dir = output_dir / 'charts'
reports_dir = output_dir / 'reports'
models_dir = output_dir / 'models'
for path in [charts_dir, reports_dir, models_dir]:
    path.mkdir(parents=True, exist_ok=True)

df = df.copy()
df = df.sort_values('timestamp').reset_index(drop=True)
df['timestamp'] = pd.to_datetime(df['timestamp'])

label_col = 'anomaly_label'
feature_columns = [c for c in df.columns if c not in ['event_id', 'timestamp', label_col]]
print('Feature columns for model:')
print(feature_columns)

missing = df[feature_columns].isna().sum()
print('Missing values in feature columns:')
print(missing[missing > 0])


Feature columns for model:
['user_id', 'database_name', 'query_type', 'table_name', 'query_length', 'rows_returned', 'duration_ms', 'bytes_transferred', 'privilege_level', 'sensitive_table', 'num_join_tables', 'error_count', 'is_successful', 'hour_of_day', 'is_weekend', 'after_hours', 'query_frequency']
Missing values in feature columns:
Series([], dtype: int64)


## 6. Feature engineering

The model uses a diverse set of features related to user activity, SQL command types, timing, query frequency, table sensitivity, and volume. The following feature engineering block composes the final feature matrix before the train/test split.


In [5]:
df['session_hour'] = df['timestamp'].dt.hour
df['day_of_week'] = df['timestamp'].dt.dayofweek
df['month'] = df['timestamp'].dt.month

df['query_type'] = df['query_type'].astype(str)
df['table_name'] = df['table_name'].astype(str)
df['user_id'] = df['user_id'].astype(str)
df['database_name'] = df['database_name'].astype(str)
df['privilege_level'] = df['privilege_level'].astype(str)

df['unusual_volume'] = (df['rows_returned'] > df['rows_returned'].quantile(0.95)).astype(int)
df['large_object_transfer'] = (df['bytes_transferred'] > df['bytes_transferred'].quantile(0.95)).astype(int)
df['high_frequency_user'] = (df['query_frequency'] > df['query_frequency'].quantile(0.95)).astype(int)
df['late_night_access'] = df['after_hours'].astype(int)
df['sensitive_access'] = df['sensitive_table'].astype(int)

feature_columns = [
    'user_id', 'database_name', 'query_type', 'table_name', 'privilege_level',
    'query_length', 'rows_returned', 'duration_ms', 'bytes_transferred',
    'num_join_tables', 'error_count', 'is_successful', 'session_hour', 'day_of_week',
    'month', 'after_hours', 'query_frequency', 'unusual_volume',
    'large_object_transfer', 'high_frequency_user', 'sensitive_access'
]
print('Final feature list:', feature_columns)


Final feature list: ['user_id', 'database_name', 'query_type', 'table_name', 'privilege_level', 'query_length', 'rows_returned', 'duration_ms', 'bytes_transferred', 'num_join_tables', 'error_count', 'is_successful', 'session_hour', 'day_of_week', 'month', 'after_hours', 'query_frequency', 'unusual_volume', 'large_object_transfer', 'high_frequency_user', 'sensitive_access']


## 7. Train/test separation

The labeled data is split into a reproducible train/test set. The model is fit on the training data only, and the hold-out test set is used for evaluation. This prevents leakage and ensures the metrics reflect the real predictive ability of the model.


In [6]:
X = df[feature_columns].copy()
y = df[label_col].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y,
)

print(f'Train rows: {len(X_train)}')
print(f'Test rows: {len(X_test)}')
print(f'Anomaly rate (train): {y_train.mean():.3f}')
print(f'Anomaly rate (test): {y_test.mean():.3f}')

train_test_summary = {
    'train_rows': int(len(X_train)),
    'test_rows': int(len(X_test)),
    'random_state': 42,
    'test_size': 0.30,
    'stratified': True,
    'anomaly_rate_train': float(y_train.mean()),
    'anomaly_rate_test': float(y_test.mean()),
}
print(json.dumps(train_test_summary, indent=2))


Train rows: 4900
Test rows: 2100
Anomaly rate (train): 0.080
Anomaly rate (test): 0.080
{
  "train_rows": 4900,
  "test_rows": 2100,
  "random_state": 42,
  "test_size": 0.3,
  "stratified": true,
  "anomaly_rate_train": 0.08,
  "anomaly_rate_test": 0.08
}


## 8. Isolation Forest training

Isolation Forest is trained with a fixed random seed and a low contamination parameter appropriate to a dataset with a small anomaly rate. This implementation uses the official scikit-learn `IsolationForest` estimator and a preprocessing pipeline for numeric and categorical variables.


In [7]:
contamination = 0.08
random_state = 42
n_estimators = 300

numeric_features = [
    'query_length', 'rows_returned', 'duration_ms', 'bytes_transferred',
    'num_join_tables', 'error_count', 'is_successful', 'session_hour', 'day_of_week',
    'month', 'after_hours', 'query_frequency', 'unusual_volume',
    'large_object_transfer', 'high_frequency_user', 'sensitive_access'
]
categorical_features = [
    'user_id', 'database_name', 'query_type', 'table_name', 'privilege_level'
]

preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric_features),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore'))
        ]), categorical_features),
    ],
    remainder='drop'
)

model = IsolationForest(
    n_estimators=n_estimators,
    contamination=contamination,
    random_state=random_state,
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model),
])

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)
model.fit(X_train_proc)

print('Model fitted successfully on training data.')
print('Training score samples summary:', np.array(model.score_samples(X_train_proc)).min(), np.array(model.score_samples(X_train_proc)).max())
print('Estimated contamination:', model.contamination)


Model fitted successfully on training data.


Training score samples summary: -0.5538542213032757 -0.38017244642560793
Estimated contamination: 0.08


## 9. Predictions and risk scoring

After fitting the model, it generates anomaly scores for the test data. `score_samples` is negated to create a higher-is-more-anomalous score, and a normalized 0-100 risk score is derived from that signal. The positive class is the anomaly class.


In [8]:
X_test_score = model.score_samples(X_test_proc)
anomaly_scores = -X_test_score
risk_scores = ((anomaly_scores - anomaly_scores.min()) / (anomaly_scores.max() - anomaly_scores.min() + 1e-8)) * 100
risk_scores = np.clip(risk_scores, 0, 100)

y_pred = (model.predict(X_test_proc) == -1).astype(int)
risk_level = np.where(risk_scores >= 70, 'High', np.where(risk_scores >= 40, 'Medium', 'Low'))

pred_df = X_test.copy().reset_index(drop=True)
pred_df['timestamp'] = df.loc[X_test.index, 'timestamp'].reset_index(drop=True).values
pred_df['true_label'] = y_test.reset_index(drop=True).to_numpy()
pred_df['predicted_label'] = y_pred
pred_df['anomaly_score'] = anomaly_scores
pred_df['risk_score'] = risk_scores.round(2)
pred_df['risk_level'] = risk_level

print('Prediction summary:')
print(pd.Series(y_pred).value_counts().to_string())
print('Risk level summary:')
print(pred_df['risk_level'].value_counts().to_string())

# Explain suspicious behavior using direct feature checks.
def explain_event(row):
    reasons = []
    if row['after_hours'] == 1:
        reasons.append('after-hours access')
    if row['sensitive_access'] == 1:
        reasons.append('sensitive-table access')
    if row['unusual_volume'] == 1:
        reasons.append('unusually large data volume')
    if row['large_object_transfer'] == 1:
        reasons.append('large bytes transferred')
    if row['high_frequency_user'] == 1:
        reasons.append('abnormal query frequency')
    if row['error_count'] > 2:
        reasons.append('multiple query errors')
    if row['rows_returned'] > df['rows_returned'].quantile(0.95):
        reasons.append('very high row count')
    if row['duration_ms'] > df['duration_ms'].quantile(0.95):
        reasons.append('extended query duration')
    if not reasons:
        return 'No strong anomaly signature detected; low-confidence deviation.'
    return '; '.join(reasons)

pred_df['explanation'] = pred_df.apply(explain_event, axis=1)
print('Sample explanations:')
print(pred_df[['timestamp', 'user_id', 'risk_level', 'explanation']].head(5).to_string(index=False))


Prediction summary:
0    1933
1     167
Risk level summary:
risk_level
Low       1933
High        84
Medium      83


Sample explanations:
          timestamp user_id risk_level                                                     explanation
2024-03-26 02:41:00 user_11        Low                                              after-hours access
2024-03-10 07:37:00 user_14        Low No strong anomaly signature detected; low-confidence deviation.
2024-01-13 23:09:00 user_06        Low                                              after-hours access
2024-01-29 06:00:00 user_08        Low                      after-hours access; sensitive-table access
2024-03-17 17:08:00 user_17        Low                                          sensitive-table access


## 10. Evaluation metrics

The model is evaluated on the held-out test set. In this synthetic-data setting, the anomaly class is considered the positive class, and metrics are computed from the true labels and predicted labels. The mapping is as follows: `IsolationForest.predict == -1` indicates anomaly; `anomaly_score` is used for ROC and PR calculations.


In [9]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, anomaly_scores)
pr_auc = average_precision_score(y_test, anomaly_scores)

tn, fp, fn, tp = cm.ravel()
fpr = fp / (fp + tn) if (fp + tn) else 0.0
fnr = fn / (fn + tp) if (fn + tp) else 0.0

metrics = {
    'dataset_name': 'synthetic_sql_logs',
    'data_source': 'synthetic generated from workspace inspection; no SQL-specific dataset found',
    'n_rows': int(len(df)),
    'n_anomalies': int(df[label_col].sum()),
    'n_normal': int((1 - df[label_col]).sum()),
    'train_rows': int(len(X_train)),
    'test_rows': int(len(X_test)),
    'random_state': random_state,
    'contamination': contamination,
    'n_estimators': n_estimators,
    'accuracy': float(accuracy),
    'precision': float(precision),
    'recall': float(recall),
    'f1_score': float(f1),
    'false_positive_rate': float(fpr),
    'false_negative_rate': float(fnr),
    'roc_auc': float(roc_auc),
    'average_precision': float(pr_auc),
    'confusion_matrix': cm.tolist(),
    'classification_report': classification_report(y_test, y_pred, target_names=['normal', 'anomaly'], output_dict=True),
}

print(json.dumps({
    'accuracy': metrics['accuracy'],
    'precision': metrics['precision'],
    'recall': metrics['recall'],
    'f1_score': metrics['f1_score'],
    'roc_auc': metrics['roc_auc'],
    'average_precision': metrics['average_precision'],
    'false_positive_rate': metrics['false_positive_rate'],
    'false_negative_rate': metrics['false_negative_rate'],
}, indent=2))

with open(reports_dir / 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

report_lines = [
    'SQL Log Anomaly Detection Evaluation Report',
    '========================================',
    f'Dataset: {metrics["dataset_name"]}',
    f'Data source: {metrics["data_source"]}',
    f'Random state: {metrics["random_state"]}',
    f'Contamination: {metrics["contamination"]}',
    f'N_estimators: {metrics["n_estimators"]}',
    f'Train rows: {metrics["train_rows"]}',
    f'Test rows: {metrics["test_rows"]}',
    '',
    f'Accuracy: {metrics["accuracy"]:.4f}',
    f'Precision: {metrics["precision"]:.4f}',
    f'Recall: {metrics["recall"]:.4f}',
    f'F1-score: {metrics["f1_score"]:.4f}',
    f'False positive rate: {metrics["false_positive_rate"]:.4f}',
    f'False negative rate: {metrics["false_negative_rate"]:.4f}',
    f'ROC-AUC: {metrics["roc_auc"]:.4f}',
    f'PR-AUC / Avg Precision: {metrics["average_precision"]:.4f}',
    '',
    'Confusion matrix (rows: true, columns: predicted):',
    str(cm),
    '',
    'Classification report:',
    classification_report(y_test, y_pred, target_names=['normal', 'anomaly']),
]
with open(reports_dir / 'evaluation_report.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(report_lines))

print(f'Wrote metrics to {reports_dir / "metrics.json"}')
print(f'Wrote evaluation report to {reports_dir / "evaluation_report.txt"}')


{
  "accuracy": 0.9938095238095238,
  "precision": 0.9640718562874252,
  "recall": 0.9583333333333334,
  "f1_score": 0.9611940298507463,
  "roc_auc": 0.999275978507345,
  "average_precision": 0.9932366957861747,
  "false_positive_rate": 0.003105590062111801,
  "false_negative_rate": 0.041666666666666664
}
Wrote metrics to C:\Users\POOJASRI M\Downloads\pavithra\pavithra\outputs\reports\metrics.json
Wrote evaluation report to C:\Users\POOJASRI M\Downloads\pavithra\pavithra\outputs\reports\evaluation_report.txt


## 11. Accuracy and precision charts

We create a simple metric comparison chart to show the model's overall performance on the test split. The chart is generated from the computed evaluation metrics and saved as a 300 DPI PNG.


In [10]:
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1-score']
metric_values = [accuracy, precision, recall, f1]

plt.figure(figsize=(8, 6))
ax = sns.barplot(x=metric_names, y=metric_values, palette='viridis')
for container in ax.containers:
    ax.bar_label(container, fmt='%.3f', padding=3)
ax.set_title('Accuracy, Precision, Recall, and F1-score on Test Set')
ax.set_ylim(0, 1.1)
plt.tight_layout()
plt.savefig(charts_dir / '01_metric_comparison.png', dpi=300)
plt.close()
print('Saved 01_metric_comparison.png')


C:\Users\POOJASRI M\AppData\Local\Temp\ipykernel_4740\2705656108.py:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x=metric_names, y=metric_values, palette='viridis')


Saved 01_metric_comparison.png


## 12. Confusion matrix and ROC/PR curves

A confusion matrix heatmap, ROC curve, and precision-recall curve provide more detail than single-value metrics, and they are all constructed directly from the model's test-set predictions.


In [11]:
cm_df = pd.DataFrame(cm, index=['Actual normal', 'Actual anomaly'], columns=['Predicted normal', 'Predicted anomaly'])
plt.figure(figsize=(7, 5))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar_kws={'label': 'Count'})
plt.title('Confusion Matrix (Anomaly = Positive Class)')
plt.tight_layout()
plt.savefig(charts_dir / '02_confusion_matrix.png', dpi=300)
plt.close()

roc_fpr, roc_tpr, _ = roc_curve(y_test, anomaly_scores)
plt.figure(figsize=(7, 5))
plt.plot(roc_fpr, roc_tpr, label=f'ROC curve (AUC={roc_auc:.3f})', linewidth=2)
plt.plot([0, 1], [0, 1], linestyle='--', color='gray', label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.tight_layout()
plt.savefig(charts_dir / '03_roc_curve.png', dpi=300)
plt.close()

pr_precision, pr_recall, _ = precision_recall_curve(y_test, anomaly_scores)
plt.figure(figsize=(7, 5))
plt.plot(pr_recall, pr_precision, label=f'PR curve (AP={pr_auc:.3f})', linewidth=2)
plt.axhline(y=y_test.mean(), color='gray', linestyle='--', label='Base rate')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.tight_layout()
plt.savefig(charts_dir / '04_precision_recall_curve.png', dpi=300)
plt.close()

print('Saved confusion matrix and ROC/PR charts.')


Saved confusion matrix and ROC/PR charts.


## 13. Explainable AI

Because Isolation Forest is an unsupervised anomaly detector, the model's output is explained with transparent, interpretable risk features rather than a claim of malicious intent. These explanations highlight unusually high frequency, unusual data volume, after-hours access, sensitive-table access, and other conditions that increase suspicion.


In [12]:
risky_rows = pred_df[pred_df['predicted_label'] == 1].copy()
if len(risky_rows) > 0:
    feature_explanations = {}
    for feature in ['query_length', 'rows_returned', 'duration_ms', 'bytes_transferred', 'query_frequency', 'error_count']:
        baseline = pred_df.loc[pred_df['predicted_label'] == 0, feature].median()
        current = risky_rows[feature].median()
        feature_explanations[feature] = abs(current - baseline) / (abs(baseline) + 1e-6)
    feature_df = pd.DataFrame({'feature': list(feature_explanations.keys()), 'deviation': list(feature_explanations.values())}).sort_values('deviation', ascending=False)

    plt.figure(figsize=(9, 6))
    sns.barplot(data=feature_df, x='deviation', y='feature', palette='rocket')
    plt.title('Feature-level explanation for suspicious events')
    plt.xlabel('Relative deviation from baseline')
    plt.ylabel('Feature')
    plt.tight_layout()
    plt.savefig(charts_dir / '09_feature_explanation.png', dpi=300)
    plt.close()

    print(feature_df.head(10).to_string(index=False))
else:
    print('No suspicious candidate rows found for explanation analysis.')


C:\Users\POOJASRI M\AppData\Local\Temp\ipykernel_4740\1141944301.py:11: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=feature_df, x='deviation', y='feature', palette='rocket')


          feature   deviation
    rows_returned 2480.329646
bytes_transferred   54.580338
      duration_ms    8.952721
     query_length    7.065779
      error_count    1.999998
  query_frequency    0.458333


## 14. Anomaly reports

The final output includes a predictions report with timestamps, anomaly scores, risk level, and transparent explanations for each suspicious event.


In [13]:
predictions_csv = reports_dir / 'predictions.csv'
pred_df_export = pred_df[['timestamp', 'user_id', 'database_name', 'query_type', 'table_name', 'query_length', 'rows_returned', 'duration_ms', 'bytes_transferred', 'privilege_level', 'query_frequency', 'after_hours', 'sensitive_access', 'error_count', 'true_label', 'predicted_label', 'anomaly_score', 'risk_score', 'risk_level', 'explanation']].copy()
pred_df_export.columns = ['timestamp', 'user_id', 'database_name', 'query_type', 'table_name', 'query_length', 'rows_returned', 'duration_ms', 'bytes_transferred', 'privilege_level', 'query_frequency', 'after_hours', 'sensitive_access', 'error_count', 'true_label', 'predicted_anomaly', 'anomaly_score', 'risk_score', 'risk_level', 'explanation']
pred_df_export.to_csv(predictions_csv, index=False)
print(f'Predictions saved to {predictions_csv}')
print(pred_df_export.head().to_string(index=False))

plt.figure(figsize=(8, 5))
sns.histplot(anomaly_scores, bins=30, kde=True, color='darkorange')
plt.axvline(np.median(anomaly_scores), color='black', linestyle='--', label=f'Median={np.median(anomaly_scores):.3f}')
plt.title('Anomaly score distribution')
plt.xlabel('Anomaly score')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.savefig(charts_dir / '05_anomaly_score_distribution.png', dpi=300)
plt.close()

plt.figure(figsize=(6, 5))
counts = pd.Series(y_pred, name='predicted_anomaly').value_counts().rename(index={0: 'Normal activity', 1: 'Anomalous activity'})
counts.plot(kind='bar', color=['steelblue', 'tomato'])
plt.title('Normal versus anomalous activity counts')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(charts_dir / '06_normal_vs_anomalous_activity.png', dpi=300)
plt.close()

plt.figure(figsize=(6, 5))
risk_counts = pred_df_export['risk_level'].value_counts().reindex(['Low', 'Medium', 'High'], fill_value=0)
risk_counts.plot(kind='bar', color=['green', 'orange', 'red'])
plt.title('Risk-level distribution')
plt.ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig(charts_dir / '07_risk_level_distribution.png', dpi=300)
plt.close()

plot_df = pred_df_export[pred_df_export['predicted_anomaly'] == 1].copy()
if not plot_df.empty:
    plot_df['timestamp'] = pd.to_datetime(plot_df['timestamp'])
    by_time = plot_df.set_index('timestamp').resample('D').size()
    plt.figure(figsize=(10, 5))
    by_time.plot(color='crimson', linewidth=2)
    plt.title('Anomalies over time')
    plt.xlabel('Date')
    plt.ylabel('Anomaly count')
    plt.tight_layout()
    plt.savefig(charts_dir / '08_anomalies_over_time.png', dpi=300)
    plt.close()

fp_fn_df = pd.DataFrame({'Metric': ['False Positive Rate', 'False Negative Rate'], 'Value': [fpr, fnr]})
plt.figure(figsize=(6, 5))
sns.barplot(data=fp_fn_df, x='Metric', y='Value', palette=['#f4a261', '#5e60ce'])
plt.title('False-positive and false-negative analysis')
plt.ylim(0, 1.1)
plt.tight_layout()
plt.savefig(charts_dir / '10_false_positive_false_negative.png', dpi=300)
plt.close()

print('Saved anomaly charts and predictions CSV.')


Predictions saved to C:\Users\POOJASRI M\Downloads\pavithra\pavithra\outputs\reports\predictions.csv
          timestamp user_id database_name query_type table_name  query_length  rows_returned  duration_ms  bytes_transferred privilege_level  query_frequency  after_hours  sensitive_access  error_count  true_label  predicted_anomaly  anomaly_score  risk_score risk_level                                                     explanation
2024-03-26 02:41:00 user_11         hr_db      MERGE   payments          1536             39          515             194986            read              238            1                 0            2           0                  0       0.399746       11.34        Low                                              after-hours access
2024-03-10 07:37:00 user_14      sales_db     SELECT   payments          2008           1153         1880             246026            read              250            0                 0            2           0                

C:\Users\POOJASRI M\AppData\Local\Temp\ipykernel_4740\2439954681.py:54: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=fp_fn_df, x='Metric', y='Value', palette=['#f4a261', '#5e60ce'])


Saved anomaly charts and predictions CSV.


## 15. Model saving and final conclusions

The fitted preprocessing pipeline and the trained Isolation Forest are saved as a reusable artifact. The final cell prints the actual metrics and the artifact paths for verification.


In [14]:
model_artifact = models_dir / 'isolation_forest.joblib'
joblib.dump({
    'model': model,
    'preprocessor': preprocessor,
    'feature_columns': feature_columns,
    'contamination': contamination,
    'random_state': random_state,
    'n_estimators': n_estimators,
    'label_column': label_col,
}, model_artifact)
print(f'Saved model artifact to {model_artifact}')

loaded = joblib.load(model_artifact)
print('Loaded model artifact successfully:', isinstance(loaded['model'], IsolationForest))

print('Actual evaluation metrics:')
print(json.dumps({
    'accuracy': float(accuracy),
    'precision': float(precision),
    'recall': float(recall),
    'f1_score': float(f1),
    'roc_auc': float(roc_auc),
    'average_precision': float(pr_auc),
    'false_positive_rate': float(fpr),
    'false_negative_rate': float(fnr),
}, indent=2))

artifact_paths = [
    str(workspace / 'SQL_Log_Anomaly_Detection.ipynb'),
    str(charts_dir / '01_metric_comparison.png'),
    str(charts_dir / '02_confusion_matrix.png'),
    str(charts_dir / '03_roc_curve.png'),
    str(charts_dir / '04_precision_recall_curve.png'),
    str(charts_dir / '05_anomaly_score_distribution.png'),
    str(charts_dir / '06_normal_vs_anomalous_activity.png'),
    str(charts_dir / '07_risk_level_distribution.png'),
    str(charts_dir / '08_anomalies_over_time.png'),
    str(charts_dir / '09_feature_explanation.png'),
    str(charts_dir / '10_false_positive_false_negative.png'),
    str(reports_dir / 'evaluation_report.txt'),
    str(reports_dir / 'metrics.json'),
    str(reports_dir / 'predictions.csv'),
    str(model_artifact),
]
print('Generated artifacts:')
for path in artifact_paths:
    print(path)

print('Conclusion:')
print('The notebook inspected the workspace, chose a synthetic SQL log dataset because no SQL-specific log dataset was present, fit an IsolationForest model, evaluated it on a reproducible hold-out set, and saved all requested outputs. This is a synthetic-data evaluation and not an assertion of malicious activity in a production system.')


Saved model artifact to C:\Users\POOJASRI M\Downloads\pavithra\pavithra\outputs\models\isolation_forest.joblib
Loaded model artifact successfully: True
Actual evaluation metrics:
{
  "accuracy": 0.9938095238095238,
  "precision": 0.9640718562874252,
  "recall": 0.9583333333333334,
  "f1_score": 0.9611940298507463,
  "roc_auc": 0.999275978507345,
  "average_precision": 0.9932366957861747,
  "false_positive_rate": 0.003105590062111801,
  "false_negative_rate": 0.041666666666666664
}
Generated artifacts:
C:\Users\POOJASRI M\Downloads\pavithra\pavithra\SQL_Log_Anomaly_Detection.ipynb
C:\Users\POOJASRI M\Downloads\pavithra\pavithra\outputs\charts\01_metric_comparison.png
C:\Users\POOJASRI M\Downloads\pavithra\pavithra\outputs\charts\02_confusion_matrix.png
C:\Users\POOJASRI M\Downloads\pavithra\pavithra\outputs\charts\03_roc_curve.png
C:\Users\POOJASRI M\Downloads\pavithra\pavithra\outputs\charts\04_precision_recall_curve.png
C:\Users\POOJASRI M\Downloads\pavithra\pavithra\outputs\charts\05